In [ ]:
# 特徴量の抽出 → ベクトルに変換 → 識別器を使って学習（機械学習）
# ニューラルネットワークは、上記のすべてをコンピュータが行う

In [ ]:
# あるデータセットだけに過度に対応した状態を過学習という

In [ ]:
# 損失関数
# ニューラルネットワークの性能の悪さを示す指標（教師データに対して、どれだけ適合できていないかを表す）
# 訓練データを使った認識精度を指標に使用しない理由は、認識精度を使用すると離散的な精度指標になってしまうため
# ニューラル
# one-hot表現：正解ラベルを1、それ以外を0で表現する表記法

In [ ]:
# 2乗和誤差

$$
E = \frac{1}{2} \sum_{k} (y_k - t_k)^2
$$

In [3]:
# 値が小さいほど、適合していると言える
import numpy as np

def sum_squared_error(y, t):
    return 0.5 * np.sum((y - t) ** 2)

y1 = [0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0] # 誤差小さい
y2 = [0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.0, 0.6, 0.0, 0.0] # 誤差大きい
t = [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]

E1 = sum_squared_error(np.array(y1), np.array(t))
E2 = sum_squared_error(np.array(y2), np.array(t))
print(E1, E2)

0.09750000000000003 0.5975


In [ ]:
# 交差エントロピー誤差

$$
E = -\sum_{k} t_k \log y_k
$$

In [11]:
# 2乗和誤差と同様に、値が小さいほど、適合していると言える
# 下記のミニバッチ学習に対応したもの
def cross_entropy_error(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)

    batch_size = y.shape[0]
    delta = 1e-7 # logの中身が発散しないようにするための補正項
    return -np.sum(t * np.log(y + delta)) / batch_size # one-hotバージョン
    # return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size # ラベルバージョン

y1 = [0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0] # 誤差小さい
y2 = [0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.0, 0.6, 0.0, 0.0] # 誤差大きい
t = [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]

E1 = cross_entropy_error(np.array(y1), np.array(t))
E2 = cross_entropy_error(np.array(y2), np.array(t))
print(E1, E2)

0.510825457099338 2.302584092994546


In [ ]:
# ミニバッチ学習
# 訓練データに対する損失関数を求め、その値をできるだけ小さくするようなパラメータを探す
# 訓練データが100個あるなら、その100個の損失関数の和を指標とする
# 膨大な訓練データから無作為に抽出し、ミニバッチとして学習を行う方法をミニバッチ学習という

$$
E = -\frac{1}{N} \sum_{n=1}^{N} \sum_{k=1}^{K} t_{nk} \log y_{nk}
$$

In [7]:
# ミニバッチの作成
x = [0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0]
batch_size = 5
x_batch = np.random.choice(x, batch_size) # ミニバッチを作成する
print(x_batch)

[0.1  0.6  0.05 0.6  0.  ]


In [14]:
# 数値微分
# hを0にすることはできないため、厳密には誤差が生まれてしまう → 中心差分を計算することで解消
# 偏微分の場合には、微分対象の変数だけを残した関数を定義し直して、数値微分をすることで計算（勾配などは別）
def numerical_diff(f, x):
    h = 1e-4 # 0.0001程度であれば、十分良い結果を得ることができる
    return (f(x+h) - f(x-h)) / (2*h)

def test_func1(x):
    return  x ** 2 + 5 * x

print(numerical_diff(test_func1, 1))
print(numerical_diff(test_func1, 2))

6.999999999992568
9.000000000014552


In [18]:
# 勾配
# 勾配が示す方向は、各場所において関数の値を最も減らす方向を指す（最小ではなく極小を指す）
# 勾配の値は、hだけ動かすと、配列*h 分だけ値が変化するという意味
def numerical_gradient(f, x):
    h = 1e-4
    grad = np.zeros_like(x) # xと同じ形状で、要素がすべて0の配列を作成

    for idx in range(x.size):
        tmp_val = x[idx]
        # f(x+h)の計算
        x[idx] = tmp_val + h
        fxh1 = f(x)
        # f(x+h)の計算
        x[idx] = tmp_val - h
        fxh2 = f(x)

        grad[idx] = (fxh1 - fxh2) / (2*h)
        x[idx] = tmp_val

    return grad

def test_func2(x):
    return x[0] ** 2 + x[1] ** 2

print(numerical_gradient(test_func2, np.array([3.0, 4.0])))
print(numerical_gradient(test_func2, np.array([-1.0, 2.0])))

[6. 8.]
[-2.  4.]


In [ ]:
# 勾配法
# 勾配を計算して、損失関数の最小値を探す方法
# 最小値を探す場合を勾配降下法、最大値を探す場合を勾配上昇法という
# 偏微分項の係数は学習率といい、ニューラルネットワークが最適化するわけではなく
# 人の手で設定する必要のある「ハイパーパラメータ」の1つ
# 学習率は大きすぎても、小さすぎても良い結果にならないので、バランスを見極める必要がある

$$
\begin{aligned}
x_0 &\leftarrow x_0 - \eta \frac{\partial f}{\partial x_0} \\
x_1 &\leftarrow x_1 - \eta \frac{\partial f}{\partial x_1}
\end{aligned}
$$

In [22]:
def gradient_descent(f, init_x, lr=0.01, step_num=100):
    x = init_x
    for i in range(step_num):
        grad = numerical_gradient(f, x)
        x -= lr * grad
    return x

print(gradient_descent(test_func2, np.array([3.0, 4.0]), lr=0.1, step_num=100))
# 学習率が大きすぎる場合
print(gradient_descent(test_func2, np.array([3.0, 4.0]), lr=10.0, step_num=100))
# 学習率が小さすぎる場合
print(gradient_descent(test_func2, np.array([3.0, 4.0]), lr=1e-9, step_num=100))

[6.11110793e-10 8.14814391e-10]
[ 2.58983747e+13 -1.29524862e+12]
[2.9999994 3.9999992]


In [67]:
from common.activation import softmax
from common.gradient import numerical_gradient

class simpleNet:
    def __init__(self):
        self.W = np.random.randn(2, 3) # ガウス分布で重みの初期化

    def predict(self, x):
        return np.dot(x, self.W)

    def loss(self, x, t):
        z = self.predict(x)
        y = softmax(z)
        loss = cross_entropy_error(y, t)
        return loss

net = simpleNet()

x = np.array([0.6, 0.9])
t = np.array([0, 0, 1])
net_weight = net.W
print(net_weight)
net_predict = net.predict(x)
print(net_predict)
print(np.argmax(net_predict))
net_loss = net.loss(x, t)
print(net_loss)

def f(W):
    return net.loss(x, t)

dW = numerical_gradient(f, net_weight)
print(dW)

[[ 0.39031413 -0.10829364  0.57509679]
 [ 0.30530494 -0.92816807  1.64105053]]
[ 0.50896292 -0.90032744  1.82200355]
2
0.2887230912519351
[[ 0.12092441  0.02954384 -0.15046825]
 [ 0.18138662  0.04431576 -0.22570238]]
